# Лабораторная работа №8: Active learning и human-in-the-loop

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Реализовать random, entropy и diversity sampling.
- Провести несколько AL-итераций.
- Построить learning curves при равном бюджете.


## Теоретическая часть

Active learning выбирает следующий batch из pool. Uncertainty ищет неуверенные объекты, diversity покрывает пространство, disagreement использует расхождение моделей. Сравнение корректно только при одинаковом annotation budget.


### Математическая постановка

Entropy sampling: $u(x)=-\sum_c p(c\mid x)\log p(c\mid x)$. Diversity можно реализовать k-center: выбрать объект, максимизирующий расстояние до уже размеченного множества.


### Материалы

- [Active Learning survey](https://arxiv.org/abs/2009.00236)
- [scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Работа выполняется на изображениях рукописных цифр `sklearn.datasets.load_digits`; oracle в этом воспроизводимом эксперименте симулирует человека, открывая истинную метку только после отбора объекта. FAST_DEV_RUN — два раунда проверки кода, полный режим — восемь раундов и повторный random baseline.

In [1]:
from pathlib import Path
import json, os, random, time, warnings
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: 2 раунда вместо 8, 1 пример на класс в initial
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def save_json(name, data):
    (ARTIFACTS / name).write_text(
        json.dumps(data, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

print(f"✔ SEED={SEED}, FAST_DEV_RUN={FAST_DEV_RUN}, ARTIFACTS={ARTIFACTS.resolve()}")

✔ SEED=42, FAST_DEV_RUN=False, ARTIFACTS=D:\ML_SOTA\ML_sota\lab8_sota\artifacts


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Стратифицированный отложенный тест

*Вес: 1 балл.*

**Постановка.** Этап «стратифицированный отложенный тест» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Разделите изображения цифр по стратифицированному правилу до выбора стратегии.
2. Сохраните и предъявите индексы train/test изображений цифр.

**Результат.** индексы train/test изображений цифр.

**Критерий принятия.** Задание считается принятым, если индексы train/test изображений цифр получены из фактических входов этого этапа и доступны для проверки.

In [2]:
# Реализуйте это задание по постановке выше.
# Артефакт: индексы train/test изображений цифр
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_all = (digits.data.astype(np.float32) / 16.0)   # значения 0..16 → 0..1
y_all = digits.target.astype(np.int64)
images_all = digits.images                        # (1797, 8, 8) для визуализации

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all, y_all, np.arange(len(y_all)),
    test_size=0.3, stratify=y_all, random_state=SEED,
)

CLASSES = list(range(10))

print(f"✔ digits: train={len(X_train)}, test={len(X_test)}, dim={X_all.shape[1]}")
print(f"✔ Классов в train: {np.bincount(y_train)}")
print(f"✔ Классов в test : {np.bincount(y_test)}")

save_json("split_indices.json", {
    "source": "sklearn.datasets.load_digits",
    "train_indices": idx_train.tolist(),
    "test_indices": idx_test.tolist(),
    "n_train": int(len(idx_train)),
    "n_test": int(len(idx_test)),
    "classes": CLASSES,
    "stratify": True,
    "random_state": SEED,
})


✔ digits: train=1257, test=540, dim=64
✔ Классов в train: [124 127 124 128 127 127 127 125 122 126]
✔ Классов в test : [54 55 53 55 54 55 54 54 52 54]


### Начальная размеченная выборка

*Вес: 1 балл.*

**Постановка.** Этап «начальная размеченная выборка» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Задайте initial с каждым классом и неразмеченный pool только из train.
2. Сохраните и предъявите непересекающиеся initial и pool.

**Результат.** непересекающиеся initial и pool.

**Критерий принятия.** Задание считается принятым, если непересекающиеся initial и pool получены из фактических входов этого этапа и доступны для проверки.

In [3]:
# Реализуйте это задание по постановке выше.
# Артефакт: непересекающиеся initial и pool
INIT_PER_CLASS = 1 if FAST_DEV_RUN else 2
rng_split = np.random.RandomState(SEED)

initial_idx = []
for c in CLASSES:
    cls_idx = np.where(y_train == c)[0]
    picked = rng_split.choice(cls_idx, size=INIT_PER_CLASS, replace=False)
    initial_idx.extend(picked.tolist())
initial_idx = np.array(sorted(initial_idx))
initial_set = set(initial_idx.tolist())
pool_idx = np.array([i for i in range(len(y_train)) if i not in initial_set])

# Проверки непересечения
assert len(set(initial_idx) & set(pool_idx)) == 0, "initial и pool пересекаются!"
assert len(initial_idx) + len(pool_idx) == len(y_train), "initial+pool ≠ train"

cls_counts_init = {int(c): int((y_train[initial_idx] == c).sum()) for c in CLASSES}
print(f"✔ Initial: {len(initial_idx)} (по классам: {cls_counts_init})")
print(f"✔ Pool: {len(pool_idx)}")

np.save(ARTIFACTS / "initial_idx.npy", initial_idx)
np.save(ARTIFACTS / "pool_idx.npy", pool_idx)
save_json("initial_pool.json", {
    "init_per_class": INIT_PER_CLASS,
    "initial_idx": initial_idx.tolist(),
    "pool_idx": pool_idx.tolist(),
    "initial_class_counts": cls_counts_init,
})

✔ Initial: 20 (по классам: {0: 2, 1: 2, 2: 2, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2})
✔ Pool: 1237


### Протокол симулированного разметчика

*Вес: 1 балл.*

**Постановка.** Этап «протокол симулированного разметчика» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните split и документируйте, что oracle открывает метку лишь после выбора.
2. Сохраните и предъявите artifacts/split.json с происхождением меток.

**Результат.** artifacts/split.json с происхождением меток.

**Критерий принятия.** Задание считается принятым, если artifacts/split.json с происхождением меток получены из фактических входов этого этапа и доступны для проверки.

In [4]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/split.json с происхождением меток
# Задание: зафиксировать split + документировать протокол oracle → artifacts/split.json.
split_doc = {
    "source": "sklearn.datasets.load_digits",
    "n_samples_total": int(len(y_all)),
    "split_rule": f"train_test_split(test_size=0.3, stratify=y_all, random_state={SEED})",
    "train_size": int(len(y_train)),
    "test_size": int(len(y_test)),
    "classes": CLASSES,
    "oracle_protocol": (
        "Oracle = симуляция человека. Истинная метка y_train[i] для i ∈ pool "
        "открывается ТОЛЬКО в момент отбора i в batch активной стратегией. "
        "До этого метки pool алгоритму недоступны. Каждый запрос фиксируется "
        "в query_log (см. learning_curves.json) как provenance."
    ),
    "label_provenance": {
        "initial": "открываются при инициализации (initial labeled set)",
        "pool":    "открываются по одному за запрос стратегии",
        "test":    "НЕ используются для обучения; только для evaluation",
    },
    "budget": {
        "init_per_class": INIT_PER_CLASS,
        "initial_size": int(len(initial_idx)),
        "pool_size":    int(len(pool_idx)),
    },
}
save_json("split.json", split_doc)
print("✔ split.json сохранён (с протоколом oracle и происхождением меток)")

✔ split.json сохранён (с протоколом oracle и происхождением меток)


### Три стратегии запроса

*Вес: 1 балл.*

**Постановка.** Этап «три стратегии запроса» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Реализуйте случайный, энтропийный и k-center выбор; в k-center обновляйте расстояния после каждого объекта.
2. Сохраните и предъявите выбор random, entropy и k-center.

**Результат.** выбор random, entropy и k-center.

**Критерий принятия.** Задание считается принятым, если выбор random, entropy и k-center получены из фактических входов этого этапа и доступны для проверки.

In [5]:
# Реализуйте это задание по постановке выше.
# Артефакт: выбор random, entropy и k-center
# Задание: реализовать random, entropy и k-center; проверить, что выбор различён и не пересекается с initial.
from scipy.spatial.distance import cdist

def query_random(pool_idx, labeled_idx, X, y, k, rng, model=None):
    return rng.choice(pool_idx, size=k, replace=False).tolist()

def query_entropy(pool_idx, labeled_idx, X, y, k, rng, model=None):
    assert model is not None, "entropy требует обученную модель"
    probs = model.predict_proba(X[pool_idx])
    eps = 1e-12
    entropy = -(probs * np.log(probs + eps)).sum(axis=1)
    order = np.argsort(-entropy)[:k]
    return [int(pool_idx[i]) for i in order]

def query_kcenter(pool_idx, labeled_idx, X, y, k, rng, model=None):
    """Greedy k-center: каждый следующий объект максимизирует расстояние до
    ближайшего размеченного; расстояния обновляются после каждого выбора."""
    pool_x    = X[pool_idx]
    labeled_x = X[labeled_idx]
    d = cdist(pool_x, labeled_x, metric="euclidean").min(axis=1)
    remaining = np.arange(len(pool_idx))
    chosen_local = []
    for _ in range(k):
        best_local = int(remaining[np.argmax(d[remaining])])
        chosen_local.append(best_local)
        # обновляем расстояния до нового выбранного
        new_d = np.linalg.norm(pool_x - pool_x[best_local], axis=1)
        d = np.minimum(d, new_d)
        remaining = remaining[remaining != best_local]
    return [int(pool_idx[i]) for i in chosen_local]

STRATEGIES = {
    "random":  query_random,
    "entropy": query_entropy,
    "kcenter": query_kcenter,
}
print("✔ Стратегии зарегистрированы:", list(STRATEGIES))

# --- Демонстрация: один batch каждой стратегией на начальном состоянии ---
demo_rng = np.random.RandomState(SEED)
demo_k = min(5, len(pool_idx))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    from sklearn.linear_model import LogisticRegression as _LR
    _demo_clf = _LR(max_iter=500, random_state=SEED).fit(X_train[initial_idx], y_train[initial_idx])

demo_results = {}
for name, fn in STRATEGIES.items():
    picked = fn(pool_idx, initial_idx, X_train, y_train, demo_k, demo_rng, model=_demo_clf)
    assert len(picked) == demo_k, f"{name}: ожидалось {demo_k} выборов"
    assert len(set(picked) & set(initial_idx.tolist())) == 0, f"{name}: пересечение с initial"
    demo_results[name] = {
        "picked_idx": [int(i) for i in picked],
        "picked_labels": [int(y_train[i]) for i in picked],
    }
    print(f"  {name:8s} → labels={demo_results[name]['picked_labels']}")

save_json("strategy_demo.json", demo_results)
print("✔ strategy_demo.json сохранён")

✔ Стратегии зарегистрированы: ['random', 'entropy', 'kcenter']
  random   → labels=[8, 0, 7, 3, 2]
  entropy  → labels=[8, 8, 8, 8, 7]
  kcenter  → labels=[7, 4, 5, 9, 1]
✔ strategy_demo.json сохранён


### Итерационное переобучение

*Вес: 1 балл.*

**Постановка.** Этап «итерационное переобучение» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. После каждого запроса дообучайте классификатор на открытых метках и фиксируйте trace.
2. Сохраните и предъявите функция run с фактическими запрошенными метками.

**Результат.** функция run с фактическими запрошенными метками.

**Критерий принятия.** Задание считается принятым, если функция run с фактическими запрошенными метками получены из фактических входов этого этапа и доступны для проверки.

In [6]:
# Реализуйте это задание по постановке выше.
# Артефакт: функция run с фактическими запрошенными метками
# Задание: функция run(strategy, ...) — цикл запрос→oracle→переобучение→метрика.
from sklearn.linear_model import LogisticRegression

def train_classifier(X_lbl, y_lbl, seed=SEED):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        clf = LogisticRegression(max_iter=2000, random_state=seed)
        clf.fit(X_lbl, y_lbl)
    return clf

def run(strategy_name, X_train, y_train, X_test, y_test,
        initial_idx, pool_idx, batch_size, n_rounds, seed=SEED):
    """Возвращает trace и полный query_log с реальными запрошенными метками."""
    rng = np.random.RandomState(seed)
    query_fn = STRATEGIES[strategy_name]
    labeled = [int(i) for i in initial_idx]
    pool    = [int(i) for i in pool_idx]

    # Инициализационное обучение (initial labels уже «открыты»)
    clf = train_classifier(X_train[labeled], y_train[labeled], seed)
    acc0 = float((clf.predict(X_test) == y_test).mean())
    trace = [{"n_labels": len(labeled), "accuracy": acc0, "round": 0}]
    query_log = []

    for r in range(1, n_rounds + 1):
        k = min(batch_size, len(pool))
        if k == 0:
            break
        picked = query_fn(pool, labeled, X_train, y_train, k, rng, model=clf)

        # Oracle открывает истинные метки ТОЛЬКО сейчас
        for i in picked:
            query_log.append({
                "idx": int(i),
                "true_label": int(y_train[i]),
                "round": r,
            })
            labeled.append(int(i))
            pool.remove(int(i))

        clf = train_classifier(X_train[labeled], y_train[labeled], seed)
        acc = float((clf.predict(X_test) == y_test).mean())
        trace.append({"n_labels": len(labeled), "accuracy": acc, "round": r})

    return {
        "strategy": strategy_name,
        "seed": seed,
        "trace": trace,
        "query_log": query_log,
        "n_final_labels": len(labeled),
        "final_accuracy": trace[-1]["accuracy"],
        "labeled_idx": labeled,
        "clf": clf,
    }

# Smoke-check
_smoke = run("random", X_train, y_train, X_test, y_test,
             initial_idx, pool_idx, batch_size=10, n_rounds=1, seed=SEED)
print(f"✔ run() работает: final_acc={_smoke['final_accuracy']:.3f}, "
      f"n_labels={_smoke['n_final_labels']}, queries={len(_smoke['query_log'])}")

✔ run() работает: final_acc=0.800, n_labels=30, queries=10


### Кривые при общем бюджете

*Вес: 1 балл.*

**Постановка.** Этап «кривые при общем бюджете» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вычислите точки кривых для одинакового числа запросов и сохраните бюджеты.
2. Сохраните и предъявите artifacts/learning_curves.json.

**Результат.** artifacts/learning_curves.json.

**Критерий принятия.** Задание считается принятым, если artifacts/learning_curves.json получены из фактических входов этого этапа и доступны для проверки.

In [7]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/learning_curves.json
# Задание: одинаковый бюджет для всех трёх стратегий → artifacts/learning_curves.json.
BATCH_SIZE = 10 if FAST_DEV_RUN else 20
N_ROUNDS   = 2  if FAST_DEV_RUN else 8

curves = {}
for name in STRATEGIES:
    res = run(name, X_train, y_train, X_test, y_test,
              initial_idx, pool_idx, BATCH_SIZE, N_ROUNDS, seed=SEED)
    curves[name] = {
        "trace":            res["trace"],
        "final_accuracy":   res["final_accuracy"],
        "n_final_labels":   res["n_final_labels"],
        "query_log":        res["query_log"],
    }
    print(f"  [{name:8s}] n_labels={res['n_final_labels']}, "
          f"final_acc={res['final_accuracy']:.3f}")

budgets = {n: c["n_final_labels"] for n, c in curves.items()}
assert len(set(budgets.values())) == 1, f"Бюджеты не равны: {budgets}"
common_budget = list(budgets.values())[0]

save_json("learning_curves.json", {
    "batch_size": BATCH_SIZE,
    "n_rounds": N_ROUNDS,
    "initial_size": int(len(initial_idx)),
    "budget_per_strategy": common_budget,
    "curves": {n: c["trace"] for n, c in curves.items()},
})
print(f"✔ learning_curves.json сохранён (общий бюджет = {common_budget} меток)")

  [random  ] n_labels=180, final_acc=0.922
  [entropy ] n_labels=180, final_acc=0.930
  [kcenter ] n_labels=180, final_acc=0.939
✔ learning_curves.json сохранён (общий бюджет = 180 меток)


### Интеграл по бюджету разметки

*Вес: 1 балл.*

**Постановка.** Этап «интеграл по бюджету разметки» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Проинтегрируйте кривые accuracy по числу полученных меток.
2. Сохраните и предъявите AUC и конечная accuracy из фактических кривых.

**Результат.** AUC и конечная accuracy из фактических кривых.

**Критерий принятия.** Задание считается принятым, если AUC и конечная accuracy из фактических кривых получены из фактических входов этого этапа и доступны для проверки.

In [8]:
# Реализуйте это задание по постановке выше.
# Артефакт: AUC и конечная accuracy из фактических кривых
# Задание: AUC = ∫ accuracy d(n_labels) / (n_max - n_min); финальная accuracy из фактических кривых.
try:
    from numpy import trapezoid as _trapz
except ImportError:
    from numpy import trapz as _trapz

def auc_curve(trace):
    xs = np.array([t["n_labels"] for t in trace], dtype=float)
    ys = np.array([t["accuracy"] for t in trace], dtype=float)
    if len(xs) < 2:
        return float(ys.mean()) if len(ys) else 0.0
    return float(_trapz(ys, xs) / (xs[-1] - xs[0]))

auc_results = {}
for name, c in curves.items():
    auc_val = auc_curve(c["trace"])
    auc_results[name] = {
        "auc":              auc_val,
        "final_accuracy":   c["final_accuracy"],
        "n_final_labels":   c["n_final_labels"],
        "trace":            c["trace"],
    }
    print(f"  [{name:8s}] AUC={auc_val:.4f}, final_acc={c['final_accuracy']:.3f}, "
          f"n_labels={c['n_final_labels']}")

save_json("auc_results.json", auc_results)
print("✔ auc_results.json сохранён")

  [random  ] AUC=0.8756, final_acc=0.922, n_labels=180
  [entropy ] AUC=0.8233, final_acc=0.930, n_labels=180
  [kcenter ] AUC=0.8812, final_acc=0.939, n_labels=180
✔ auc_results.json сохранён


### Повторные случайные запросы

*Вес: 1 балл.*

**Постановка.** Этап «повторные случайные запросы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Повторите random с разными генераторами при неизменном train/test.
2. Сохраните и предъявите результаты random при различных seed.

**Результат.** результаты random при различных seed.

**Критерий принятия.** Задание считается принятым, если результаты random при различных seed получены из фактических входов этого этапа и доступны для проверки.

In [9]:
# Реализуйте это задание по постановке выше.
# Артефакт: результаты random при различных seed
# Задание: random с несколькими seed при неизменном train/test.
SEEDS = [SEED, SEED + 1, SEED + 2]
if not FAST_DEV_RUN:
    SEEDS += [SEED + 3, SEED + 4]

random_runs = []
for s in SEEDS:
    res = run("random", X_train, y_train, X_test, y_test,
              initial_idx, pool_idx, BATCH_SIZE, N_ROUNDS, seed=s)
    auc_val = auc_curve(res["trace"])
    random_runs.append({
        "seed": s,
        "auc": auc_val,
        "final_accuracy": res["final_accuracy"],
        "trace": res["trace"],
    })
    print(f"  random seed={s}: AUC={auc_val:.4f}, final_acc={res['final_accuracy']:.3f}")

aucs = [r["auc"] for r in random_runs]
accs = [r["final_accuracy"] for r in random_runs]

save_json("random_repeats.json", {
    "n_seeds": len(SEEDS),
    "seeds": SEEDS,
    "batch_size": BATCH_SIZE,
    "n_rounds": N_ROUNDS,
    "auc_mean": float(np.mean(aucs)),
    "auc_std":  float(np.std(aucs)),
    "final_acc_mean": float(np.mean(accs)),
    "final_acc_std":  float(np.std(accs)),
    "runs": random_runs,
})
print(f"✔ random_repeats.json: AUC={np.mean(aucs):.4f} ± {np.std(aucs):.4f} "
      f"по {len(SEEDS)} seed")

  random seed=42: AUC=0.8756, final_acc=0.922
  random seed=43: AUC=0.8576, final_acc=0.891
  random seed=44: AUC=0.8764, final_acc=0.935
  random seed=45: AUC=0.8784, final_acc=0.922
  random seed=46: AUC=0.8638, final_acc=0.904
✔ random_repeats.json: AUC=0.8703 ± 0.0082 по 5 seed


### Разбор ошибок отложенного теста

*Вес: 1 балл.*

**Постановка.** Этап «разбор ошибок отложенного теста» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните изображения и истинные/предсказанные классы ошибочных test-примеров.
2. Сохраните и предъявите artifacts/test_errors.json с неверно распознанными изображениями.

**Результат.** artifacts/test_errors.json с неверно распознанными изображениями.

**Критерий принятия.** Задание считается принятым, если artifacts/test_errors.json с неверно распознанными изображениями получены из фактических входов этого этапа и доступны для проверки.

In [10]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/test_errors.json с неверно распознанными изображениями
# Задание: сохранить изображения + истинные/предсказанные классы ошибочных test.
from PIL import Image

# Лучшая стратегия по AUC на сохранённых предсказаниях
best_name = max(auc_results, key=lambda n: auc_results[n]["auc"])
print(f"Лучшая стратегия по AUC: {best_name}")

best_run = run(best_name, X_train, y_train, X_test, y_test,
               initial_idx, pool_idx, BATCH_SIZE, N_ROUNDS, seed=SEED)
clf_best = best_run["clf"]
test_pred  = clf_best.predict(X_test)
test_proba = clf_best.predict_proba(X_test)
err_mask   = test_pred != y_test
err_pos    = np.where(err_mask)[0]

ERR_DIR = ARTIFACTS / "test_errors"
ERR_DIR.mkdir(parents=True, exist_ok=True)

test_errors = []
for i in err_pos:
    img = images_all[idx_test[i]]                      # (8, 8), 0..16
    img_u8 = (img / max(img.max(), 1) * 255).astype(np.uint8)
    fpath = ERR_DIR / f"err_{i:04d}_true{int(y_test[i])}_pred{int(test_pred[i])}.png"
    Image.fromarray(img_u8).resize((64, 64), Image.NEAREST).save(fpath)
    test_errors.append({
        "test_pos":         int(i),
        "original_index":   int(idx_test[i]),
        "true_label":       int(y_test[i]),
        "predicted_label":  int(test_pred[i]),
        "confidence":       float(test_proba[i, test_pred[i]]),
        "image_path":       str(fpath),
    })

save_json("test_errors.json", {
    "strategy":       best_name,
    "n_test":         int(len(y_test)),
    "n_errors":       int(err_mask.sum()),
    "error_rate":     float(err_mask.mean()),
    "errors":         test_errors,
})
print(f"✔ test_errors.json: {len(test_errors)}/{len(y_test)} "
      f"({err_mask.mean():.2%})")

Лучшая стратегия по AUC: kcenter
✔ test_errors.json: 33/540 (6.11%)


### Сводка сравнительного эксперимента

*Вес: 1 балл.*

**Постановка.** Этап «сводка сравнительного эксперимента» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сопоставьте AUC и конечную точность по трём стратегиям.
2. Сохраните и предъявите artifacts/comparison.json.

**Результат.** artifacts/comparison.json.

**Критерий принятия.** Задание считается принятым, если artifacts/comparison.json получены из фактических входов этого этапа и доступны для проверки.

In [11]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/comparison.json
# Задание: artifacts/comparison.json — AUC + финальная точность по трём стратегиям.
comparison = {
    "budget": {
        "batch_size":    BATCH_SIZE,
        "n_rounds":      N_ROUNDS,
        "initial_size":  int(len(initial_idx)),
        "total_labels":  int(curves["random"]["n_final_labels"]),
    },
    "strategies": {},
    "random_baseline_repeats": {
        "n_seeds": len(SEEDS),
        "auc_mean": float(np.mean(aucs)),
        "auc_std":  float(np.std(aucs)),
        "final_acc_mean": float(np.mean(accs)),
        "final_acc_std":  float(np.std(accs)),
    },
}

for name in ["random", "entropy", "kcenter"]:
    comparison["strategies"][name] = {
        "auc":            auc_results[name]["auc"],
        "final_accuracy": auc_results[name]["final_accuracy"],
        "n_final_labels": auc_results[name]["n_final_labels"],
    }

ranking = sorted(comparison["strategies"].items(), key=lambda kv: -kv[1]["auc"])
comparison["ranking_by_auc"] = [
    {"rank": i + 1, "strategy": name, **vals}
    for i, (name, vals) in enumerate(ranking)
]

save_json("comparison.json", comparison)
print("✔ comparison.json сохранён")
print("  Рейтинг по AUC:")
for r in comparison["ranking_by_auc"]:
    print(f"   {r['rank']}. {r['strategy']:8s}  AUC={r['auc']:.4f}  "
          f"final_acc={r['final_accuracy']:.3f}")

✔ comparison.json сохранён
  Рейтинг по AUC:
   1. kcenter   AUC=0.8812  final_acc=0.939
   2. random    AUC=0.8756  final_acc=0.922
   3. entropy   AUC=0.8233  final_acc=0.930


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [12]:
# Реализуйте творческое мини-исследование по постановке выше.
# Задание: проверяемая гипотеза о предметном ограничении метода + проверка на сохранённых
# предсказаниях (query_log), без использования test для выбора гипотезы → artifacts/bonus.md.
bonus_stats = {}
for name in ["random", "entropy", "kcenter"]:
    ql = curves[name]["query_log"]
    per_class = np.zeros(10, dtype=int)
    for q in ql:
        per_class[q["true_label"]] += 1
    p = per_class / max(per_class.sum(), 1)
    H = float(-(p * np.log(p + 1e-12)).sum())
    bonus_stats[name] = {
        "per_class_queries": per_class.tolist(),
        "n_queries":         int(per_class.sum()),
        "class_entropy":     H,
    }

# Гипотеза: k-center покрывает классы равномернее (H выше) и это согласуется с AUC.
kc_H = bonus_stats["kcenter"]["class_entropy"]
en_H = bonus_stats["entropy"]["class_entropy"]
rn_H = bonus_stats["random"]["class_entropy"]

if kc_H > en_H:
    conclusion = (
        f"Гипотеза подтверждена: энтропия распределения запрошенных классов у k-center "
        f"H={kc_H:.3f} выше, чем у entropy H={en_H:.3f}. Diversity-стратегия "
        f"покрывает классы равномернее. Это согласуется с AUC "
        f"({auc_results['kcenter']['auc']:.4f} vs {auc_results['entropy']['auc']:.4f})."
    )
else:
    conclusion = (
        f"Гипотеза НЕ подтверждена: entropy запросил не менее разнообразные классы "
        f"(H={en_H:.3f}) чем k-center (H={kc_H:.3f}). Наблюдаемая разница AUC "
        f"({auc_results['kcenter']['auc']:.4f} vs {auc_results['entropy']['auc']:.4f}) "
        f"может объясняться малым бюджетом и линейным классификатором."
    )

bonus_md = f"""# Творческое мини-исследование (Лабораторная №8)

## Гипотеза
При одинаковом бюджете запросов k-center diversity-стратегия выбирает более равномерно
разные классы, чем uncertainty (entropy). Это проявляется в более высокой энтропии
распределения запрошенных классов и должно коррелировать с AUC, если классы
несбалансированно представлены в initial.

## Протокол
- Источник: `query_log` из `artifacts/learning_curves.json` (те же запросы, что в
  основном эксперименте; отложенный тест для формулировки/выбора гипотезы НЕ использовался).
- Для каждой стратегии считаем распределение запрошенных меток по 10 классам
  и его нормированную энтропию H = -Σ p_c log p_c (равномерное = log 10 ≈ 2.303).
- Сравниваем H(k-center) vs H(entropy) и сопоставляем с AUC/финальной точностью из
  `artifacts/comparison.json`.

## Наблюдения

| Стратегия | H(class distribution) | AUC | final_acc |
|-----------|-----------------------|-----|-----------|
| random    | {rn_H:.4f} | {auc_results['random']['auc']:.4f} | {auc_results['random']['final_accuracy']:.3f} |
| entropy   | {en_H:.4f} | {auc_results['entropy']['auc']:.4f} | {auc_results['entropy']['final_accuracy']:.3f} |
| k-center  | {kc_H:.4f} | {auc_results['kcenter']['auc']:.4f} | {auc_results['kcenter']['final_accuracy']:.3f} |

Per-class распределение запросов:
- random:   {bonus_stats['random']['per_class_queries']}
- entropy:  {bonus_stats['entropy']['per_class_queries']}
- k-center: {bonus_stats['kcenter']['per_class_queries']}

## Вывод
{conclusion}

> FAST_DEV_RUN={'ON' if FAST_DEV_RUN else 'OFF'}, batch_size={BATCH_SIZE}, n_rounds={N_ROUNDS}.
> Сравнение воспроизводимо: все запросы зафиксированы в learning_curves.json.
"""
(ARTIFACTS / "bonus.md").write_text(bonus_md, encoding="utf-8")
save_json("bonus.json", {"hypothesis_stats": bonus_stats,
                          "conclusion": conclusion,
                          "fast_dev_run": FAST_DEV_RUN})
print("✔ bonus.md и bonus.json сохранены")
print(f"  H(random)={rn_H:.3f}, H(entropy)={en_H:.3f}, H(kcenter)={kc_H:.3f}")

✔ bonus.md и bonus.json сохранены
  H(random)=2.298, H(entropy)=2.253, H(kcenter)=2.233


## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.